# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import sys
import os

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

## Test de Originación y Amortización
En esta prueba vamos a:
1. Crear un **Socio Comercial** y una **Cartera**.
2. Dar de alta un **Cliente** de prueba.
3. Originar un **Crédito** con una TNA que incluya IVA.
4. Generar las **Cuotas** automáticamente usando nuestro `AmortizationEngine`.

In [ ]:
"""
Notebook Cell: Final Logic Integration Test
Description: Validates amortization calculation and dynamic 'origen' property.
Author: Juan Martín Carini
Date: 2026-05-11
"""

import datetime  # noqa: E402
import pandas as pd  # noqa: E402
import src.database  # noqa: E402
import src.logic.amortization  # noqa: E402
from src.database.connection import engine  # noqa: E402

'''
db = src.database.SessionLocal()

try:
    # 1. Preparar Entidades de prueba
    socio = src.database.SocioComercial(razon_social="Mutual del Sur", cuit="30777888991", dia_corte=13)
    db.add(socio)
    db.flush()

    # Escenario: Crédito COMPRADO (asignado a una cartera)
    cartera = src.database.Cartera(nombre="Fideicomiso Mayo 2026", socio_id=socio.id, fecha_compra=datetime.date.today(), tna_descuento=0.45)
    db.add(cartera)
    db.flush()

    cliente = src.database.Cliente(cuil="20445556661", documento="44555666", apellido="Martínez", nombre="Luis", sexo=src.database.SexoEnum.MASCULINO)
    db.add(cliente)
    db.flush()

    # 2. Originar Crédito (Simulamos una compra de cartera)
    # Al asignar cartera_id, la propiedad @hybrid_property debe devolver COMPRADO
    nuevo_credito = src.database.Credito(
        cliente_cuil=cliente.cuil,
        cartera_id=cartera.id,
        socio_originador_id=socio.id,
        capital=150000.0,
        tna_c_iva=0.75,
        plazo=12,
        fecha_emision=datetime.date.today(),
    )
    db.add(nuevo_credito)
    db.flush()

    # 3. Validar Propiedad Dinámica antes de persistir cuotas
    print(f"ID Crédito: {nuevo_credito.id}")
    print(f"Origen detectado (Hybrid Property): {nuevo_credito.origen.value}") # Debería ser COMPRADO

    # 4. Generar Cuotas con el motor de lógica (numpy_financial)
    cuotas = src.logic.amortization.AmortizationEngine.generate_french_schedule(
        credito_id=nuevo_credito.id,
        capital=nuevo_credito.capital,
        tna_c_iva=nuevo_credito.tna_c_iva,
        plazo=nuevo_credito.plazo,
        gracia=2,
        fecha_emision=nuevo_credito.fecha_emision,
        dia_corte=socio.dia_corte
    )
    db.add_all(cuotas)
    db.commit()

    # 5. Visualización de Resultados
    # Verificamos que las cuotas vencen el 28 y los montos son razonables
    query = f"SELECT * FROM cuotas WHERE credito_id = {nuevo_credito.id}"
    df_resultado = pd.read_sql(query, db.bind)
    
    # Añadimos una columna de control para ver la cuota total (PMT)
    df_resultado['total_pmt'] = df_resultado['capital'] + df_resultado['interes'] + df_resultado['iva_interes']
    
    display(df_resultado[['nro_cuota', 'fecha_vencimiento', 'capital', 'interes', 'iva_interes', 'total_pmt']])

except Exception as e:
    db.rollback()
    print(f"❌ Error en el test: {e}")
finally:
    db.close()
'''

# 🚀 Ejecución del Pipeline ETL con Archivos Reales

En esta sección, pondremos en marcha el motor de ingesta de datos (`PortfolioImporter`) utilizando los archivos físicos extraídos del sistema origen. 

Este proceso realiza las siguientes operaciones en memoria utilizando **Pandas** antes de interactuar con la base de datos:
1. **Inyección de Cabeceras:** Asigna los nombres de columnas correspondientes a los CSV crudos.
2. **Limpieza Vectorizada:** Formatea los CUILs, limpia caracteres especiales (soportando codificación `latin-1`) y estandariza fechas.
3. **Validación de Integridad:** Verifica que no existan créditos huérfanos sin un `ID Operación` válido en el archivo de personas.

#### 🛡️ Seguridad Transaccional (ACID)
El bloque de carga a la base de datos está encapsulado en una **transacción atómica**. Esto garantiza que si el script encuentra una inconsistencia en la fila 10.000 del archivo de cuotas, se ejecutará un *Rollback* automático. La base de datos volverá a su estado original, evitando que queden registros cargados por la mitad.

> **⚠️ Instrucciones antes de ejecutar:** > Verificá que las rutas absolutas en la variable `ruta_personas`, `ruta_prestamos` y `ruta_cuotas` apunten correctamente a la ubicación de los archivos en tu disco local.

In [ ]:
"""
Notebook Cell: Insert Dummy Partner
Description: Inserts a placeholder commercial partner with ID 1 to satisfy Foreign Key constraints.
"""

from src.database.connection import SessionLocal  # noqa: E402
from src.database.models import SocioComercial  # noqa: E402

db = SessionLocal()

try:
    # Check if the dummy partner already exists to avoid UniqueConstraint errors
    socio_ficticio = db.query(SocioComercial).filter_by(id=1).first()
    
    if not socio_ficticio:
        nuevo_socio = SocioComercial(
            id=1,
            razon_social="SOCIO ORIGINADOR FICTICIO",
            cuit="00000000000"  # CUIT genérico o nulo según permita tu modelo
        )
        db.add(nuevo_socio)
        db.commit()
        print("Socio ficticio creado exitosamente con ID 1.")
    else:
        print("El socio ficticio con ID 1 ya existe en la base de datos.")

except Exception as e:
    db.rollback()
    print(f"Error al insertar el socio ficticio: {e}")
finally:
    db.close()

In [ ]:
"""
Notebook Cell: Data Validation and Audit Report Generation
Description: Validates data consistency, recalculates VA, and generates an Excel report with highlighted errors.
Author: Juan Martín Carini
Date: 2026-05-12
"""

from datetime import date # noqa: E402
from importlib import reload # noqa: E402
import src.etl.csv_importer as csv_importer # noqa: E402

reload(csv_importer)
    
# 1. Definir rutas reales
ruta_personas = r"D:\Repositorios\Credit_Manager\data\PERSONAS.CSV"
ruta_prestamos = r"D:\Repositorios\Credit_Manager\data\PRESTAMOS.CSV"
ruta_cuotas = r"D:\Repositorios\Credit_Manager\data\CUOTAS.CSV"

# Read the file
cols_prestamos = [
            "ID Operación",
            "ID Entidad",
            "ID Tipo Operación",
            "Importe Cuota",
            "Capital",
            "Interés",
            "IVA",
            "Capital Vendido",
            "Interés Vendido",
            "IVA Vendido",
            "Fecha Compra",
            "Tasa Compra",
            "Valor Actual",
            "Ente Pagador",
            "CBU/CVU",
        ]

df_raw = pd.read_csv(
    ruta_prestamos,
    sep=",",
    encoding="latin-1",
    header=None,
    names=cols_prestamos,
)
# Update the column
df_raw["ID Entidad"] = 1

# Save the patched file over the original
df_raw.to_csv(ruta_prestamos, sep=",", index=False, header=False)
print("Archivo CSV actualizado con ID 1.")


print("Iniciando proceso...")

try:
    # 2. Instanciar el objeto importador (esto ejecuta el __init__ y abre la sesión de BD)
    importer = csv_importer.PortfolioImporter()

    # 3. Pre-crear la Cartera y el Socio (OBLIGATORIO: Guarda el TNA y Fecha en memoria para la validación)
    importer.create_portfolio(
        nombre_cartera="Compra Mayo 2026",
        fecha_compra=date(2026, 3, 10),
        tna_descuento=0.40,
        cuit_vendedor="30712345678",
        razon_social_vendedor="Proveedor Ejemplo"
    )

    # 4. Extracción de datos
    importer.read_csv(
        personas_path=ruta_personas,
        prestamos_path=ruta_prestamos,
        cuotas_path=ruta_cuotas
    )

    if importer.data_loaded:
        importer.validation()
        importer.check_warnings()
        importer.save_portfolio()

except Exception as e:
    print(f"\n❌ Error en la ejecución: {e}")

In [ ]:
from src.reports import saldos

# Ejemplo
df = saldos(con_saldo=False, propias=False, agrupar=True, socios=False, originador=False, vencimientos=True, clientes=False, dueño=True)

display(df)